# kaggle

> Jobs as Kaggle kernels through the `kaggle` CLI: a CPU, two T4s, an L4, an A100, an H100 or an RTX PRO 6000, or a TPU v5e-8 or v6e-8; local data as private Kaggle datasets; the kernel's log followed live; and Kaggle Models for publishing.

In [ ]:
#| default_exp kaggle

In [ ]:
#| export
import json, os, re, shutil, subprocess, sys, tempfile, time, warnings
from pathlib import Path

from sysone.cloud import (Kaggle, CPU, GPU, TPU, Job, LocalPath, KaggleData, HubData, machine, stage, submit, poll, cancel,
                          collect, pack, runner_source, unpack_outputs, read_from, save_job, _alive)

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
from sysone.cloud import remote, jobs
from sysone.inference import Decider

A Kaggle kernel runs one uploaded file, so a sysone job on Kaggle is the [runner](40_cloud.ipynb#the-jobs-files) with the job's folder embedded in it: `kaggle kernels push` uploads that file with a `kernel-metadata.json` (the accelerator, internet on, the datasets to mount), and the kernel unpacks the job into `/kaggle/working/sysone_job` and runs it. Its log is followed live (`kaggle kernels logs -f`, in a background process writing the job's log here), its status polled (`kaggle kernels status`), and at the end `kaggle kernels output` brings `outputs.tar.gz` back.

What Kaggle does not allow shapes the rest:

- **No secrets reach a pushed kernel**: Kaggle's secrets are attached to a notebook in its editor, and the CLI has no way to pass one. So a job with `push=` does not push from Kaggle; `job.fetch()` pushes the export from here, with your Hub login. For the same reason a job cannot read a private Hub dataset; local data goes to Kaggle as a private Kaggle dataset instead.
- **No way to stop a kernel from the CLI**: `job.stop()` gives the kernel's page, where Kaggle's Stop button is.
- **Quotas and limits**: about 30 GPU hours and 20 TPU hours a week, 12 hours per session (9 on TPUs), 20 GB of outputs. Accounts need a verified phone number for internet access, which the job uses to install sysone.

In [ ]:
#| export
class KaggleError(RuntimeError): "The Kaggle CLI reported a failure"

def run_kaggle(args:list, timeout:float=600, check:bool=True) -> str:
    "Run a `kaggle` command and return its output; Kaggle reports some failures with exit code 0, so its output is read too"
    if shutil.which("kaggle") is None: raise KaggleError("the Kaggle CLI is not installed: uv tool install kaggle")
    try: r = subprocess.run(["kaggle", *[str(a) for a in args]], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: raise KaggleError(f"kaggle {' '.join(map(str, args[:3]))} took over {timeout:.0f} s") from None
    out = r.stdout + r.stderr
    if check and (r.returncode != 0 or re.search(r"(push|creation) error|error:|failed", r.stdout, re.I)):
        raise KaggleError(f"kaggle {' '.join(map(str, args[:3]))} failed ({r.returncode}): {out.strip()[-1500:]}")
    return out

def kaggle_owner(b:Kaggle) -> str:
    "The account jobs go under: the backend's owner, KAGGLE_USERNAME, or the CLI's logged-in user"
    if b.owner: return b.owner
    if os.environ.get("KAGGLE_USERNAME"): return os.environ["KAGGLE_USERNAME"]
    from sysone.auth import kaggle_user
    u = kaggle_user()
    if not u: raise KaggleError("no Kaggle account: log in (sysone.auth.login('kaggle')) or pass Kaggle(owner=...)")
    return u

## Hardware and data

Kaggle names its accelerators itself; a T4 machine always has two (`GPU("T4", 2)`, which trains with both through `torchrun`). As of September 2026 the P100 is gone, and TPU pushes from the CLI may get a machine without a TPU (Kaggle's issue 1197), which the runner's `start` line shows.

In [ ]:
#| export
KAGGLE_GPUS = {"T4": "NvidiaTeslaT4", "L4": "NvidiaL4X1", "A100": "NvidiaTeslaA100", "H100": "NvidiaH100", "RTX6000": "NvidiaRtxPro6000"}
KAGGLE_TPUS = {"v5e-8": "TpuV5E8", "v6e-8": "TpuV6E8"}

@machine.register
def machine_kaggle_cpu(b:Kaggle, hw:CPU) -> str: return "cpu"

@machine.register
def machine_kaggle_gpu(b:Kaggle, hw:GPU) -> str:
    if hw.name not in KAGGLE_GPUS: raise ValueError(f"Kaggle's GPUs are {', '.join(KAGGLE_GPUS)}, not {hw.name}")
    if hw.count > (2 if hw.name == "T4" else 1): raise ValueError(f"Kaggle has at most {2 if hw.name == 'T4' else 1} {hw.name}")
    return KAGGLE_GPUS[hw.name]

@machine.register
def machine_kaggle_tpu(b:Kaggle, hw:TPU) -> str:
    if hw.kind not in KAGGLE_TPUS: raise ValueError(f"Kaggle's TPUs are {', '.join(KAGGLE_TPUS)}, not {hw.kind}")
    warnings.warn("TPU kernels pushed from the CLI can start without a TPU (Kaggle issue 1197); check the job's start line")
    return KAGGLE_TPUS[hw.kind]

In [ ]:
test_eq([machine(Kaggle(), h) for h in (CPU(), GPU("T4", 2), GPU("L4"), GPU("H100"))], ["cpu", "NvidiaTeslaT4", "NvidiaL4X1", "NvidiaH100"])
test_fail(lambda: machine(Kaggle(), GPU("G4")), contains="Kaggle's GPUs")
test_fail(lambda: machine(Kaggle(), TPU("v5e-1")), contains="Kaggle's TPUs")

Data reaches a kernel in three ways: a Kaggle dataset is mounted under `/kaggle/input` (the runner finds it there by name), local data up to `KAGGLE_SHIP_MB` travels inside the kernel's file, and larger local data becomes a private Kaggle dataset first (`kaggle_dataset`). A public Hub dataset is downloaded by the job.

In [ ]:
#| export
KAGGLE_SHIP_MB = 20

def _slug(s:str) -> str: return re.sub(r"[^a-z0-9-]+", "-", s.lower()).strip("-")[:50]

@stage.register
def stage_kaggle_dataset(d:KaggleData, b:Kaggle, var:str) -> dict:
    return {"kaggle_datasets": [d.ref], "find": {var: f"/kaggle/input/**/{d.ref.split('/')[-1]}"}}

@stage.register
def stage_kaggle_local(d:LocalPath, b:Kaggle, var:str) -> dict:
    p = Path(d.path)
    size = p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    if size <= KAGGLE_SHIP_MB * 2**20: return {"ship": {var: d.path}}
    ref = kaggle_dataset(p, f"{kaggle_owner(b)}/{_slug('sysone-' + p.stem)}")
    return stage_kaggle_dataset(KaggleData(ref), b, var)

def kaggle_dataset(path, ref:str, private:bool=True, dry_run:bool=False) -> str:
    "Upload a file or folder as a Kaggle dataset `owner/slug` (a new version when it exists); returns its ref"
    path, folder = Path(path), Path(tempfile.mkdtemp(prefix="sysone-kaggle-data-"))
    if path.is_dir(): shutil.copytree(path, folder, dirs_exist_ok=True)
    else: shutil.copy2(path, folder / path.name)
    (folder / "dataset-metadata.json").write_text(json.dumps({"title": ref.split("/")[1], "id": ref, "licenses": [{"name": "other"}]}))
    if dry_run: return ref
    try: run_kaggle(["datasets", "version", "-p", folder, "-m", "sysone", "-r", "zip", "-q"], timeout=3600)
    except KaggleError as e:                 # not there yet: create it
        if not re.search(r"not found|does not exist|404|403", str(e), re.I): raise
        run_kaggle(["datasets", "create", "-p", folder, "-r", "zip", "-q"] + ([] if private else ["-u"]), timeout=3600)
    return ref

In [ ]:
test_eq(stage(KaggleData("me/emotions"), Kaggle(), "SYSONE_DATA"), {"kaggle_datasets": ["me/emotions"], "find": {"SYSONE_DATA": "/kaggle/input/**/emotions"}})
test_eq(list(stage(LocalPath(str(Path("fixtures/typed_decisions_tiny.jsonl").resolve())), Kaggle(), "SYSONE_DATA")), ["ship"])

## Starting and following a kernel

In [ ]:
#| export
KAGGLE_DIR = "/kaggle/working/sysone_job"

def kernel_folder(b:Kaggle, job:Job) -> Path:
    "The folder `kaggle kernels push` uploads: the runner with the job embedded, and kernel-metadata.json"
    folder = Path(job.folder)
    spec = json.loads((folder / "job.json").read_text())
    spec["push"] = None                                    # no Hub token on Kaggle: job.fetch() pushes from here
    (folder / "job.json").write_text(json.dumps(spec, indent=1))
    kf = folder / "kernel"; kf.mkdir(exist_ok=True)
    (kf / "sysone_job.py").write_text(runner_source(KAGGLE_DIR, pack(folder, skip=("sysone_job.py",))))
    hw = job.machine
    meta = {"id": job.ref, "title": job.ref.split("/")[1], "code_file": "sysone_job.py", "language": "python", "kernel_type": "script",
            "is_private": b.private, "enable_gpu": hw.startswith("Nvidia"), "enable_tpu": hw.startswith("Tpu"), "enable_internet": b.internet,
            "dataset_sources": list(spec.get("kaggle_datasets") or []), "competition_sources": [], "kernel_sources": [], "model_sources": []}
    if hw != "cpu": meta["machine_shape"] = hw
    (kf / "kernel-metadata.json").write_text(json.dumps(meta, indent=1))
    return kf

def _follow(job:Job):
    "Follow the kernel's log live into the job's log here (a background process)"
    log = open(Path(job.folder) / "job.log", "ab")
    env = dict(os.environ, PYTHONIOENCODING="latin-1:backslashreplace")     # the CLI reads the stream as Latin-1: written back as such, it is UTF-8 again
    p = subprocess.Popen(["kaggle", "kernels", "logs", "-f", job.ref], stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=env)
    (Path(job.folder) / "follower.pid").write_text(str(p.pid))

def parse_kaggle_status(text:str) -> str:
    "The status word in `kaggle kernels status` output: queued, running, complete, error or cancelacknowledged"
    m = re.search(r'has status "([^"]+)"', text)
    if not m: raise KaggleError(f"unrecognised status output: {text[:200]!r}")
    return m.group(1).split(".")[-1].replace("_", "").lower()

@submit.register
def submit_kaggle(b:Kaggle, job:Job) -> Job:
    job.ref = f"{kaggle_owner(b)}/{_slug(job.id)}"
    kf = kernel_folder(b, job)
    spec = json.loads((Path(job.folder) / "job.json").read_text())
    acc = ["--accelerator", job.machine] if job.machine != "cpu" else []
    t = ["-t", str(spec["max_seconds"])] if spec.get("max_seconds") else []
    run_kaggle(["kernels", "push", "-p", kf] + acc + t, timeout=900)
    job.status = "running"
    _follow(job)
    return job

@poll.register
def poll_kaggle(b:Kaggle, job:Job) -> str:
    st = parse_kaggle_status(run_kaggle(["kernels", "status", job.ref]))
    folder = Path(job.folder)
    pid = (folder / "follower.pid").read_text() if (folder / "follower.pid").exists() else None
    if st in ("queued", "running") and not _alive(pid): _follow(job)      # the follower stopped early: follow again
    text, job.offset = read_from(folder / "job.log", job.offset)
    if st in ("complete", "error", "cancelacknowledged"):
        out = folder / "kaggle-output"
        run_kaggle(["kernels", "output", job.ref, "-p", out, "-o", "-q", "--file-pattern", r"sysone_job/outputs\.tar\.gz"], timeout=3600, check=False)
        found = sorted(out.rglob("outputs.tar.gz"))
        if found:
            shutil.move(str(found[0]), folder / "outputs.tar.gz")
            import tarfile
            with tarfile.open(folder / "outputs.tar.gz") as t:
                s = json.load(t.extractfile("outputs/status.json"))
            job.status = s["status"]
        else: job.status = "stopped" if st == "cancelacknowledged" else "error"
    return text

@cancel.register
def cancel_kaggle(b:Kaggle, job:Job) -> Job:
    raise KaggleError(f"Kaggle's CLI cannot stop a kernel: stop it at https://www.kaggle.com/code/{job.ref} (the Stop button of its session)")

@collect.register
def collect_kaggle(b:Kaggle, job:Job, dest:Path) -> Path:
    tb = Path(job.folder) / "outputs.tar.gz"
    if not tb.exists(): raise KaggleError(f"{job.id} is {job.status}: its outputs come back when it ends (job.wait())")
    return unpack_outputs(tb, dest)

In [ ]:
test_eq(parse_kaggle_status('me/k has status "KernelWorkerStatus.COMPLETE"'), "complete")
test_eq(parse_kaggle_status('me/k has status "cancelAcknowledged"'), "cancelacknowledged")

## A job on a stand-in Kaggle

As for Colab, a stand-in for the `kaggle` CLI runs the whole round trip here: it runs a pushed kernel's file with `/kaggle` mapped to a temporary folder, reports its status, streams its log and hands out its outputs.

In [ ]:
fake_bin, fake_kaggle = Path(tempfile.mkdtemp()), Path(tempfile.mkdtemp())
(fake_bin / "kaggle").write_text(f"""#!{sys.executable}
import glob, json, os, shutil, subprocess, sys, time
root = os.environ["FAKE_KAGGLE_ROOT"]
a = sys.argv[1:]
open(os.path.join(root, "calls.log"), "a").write(json.dumps(a) + "\\n")
def opt(name, default=None): return a[a.index(name) + 1] if name in a else default
def state(ref): return os.path.join(root, ref.replace("/", "__"))
if a[:2] == ["config", "view"]: print("username: me")
elif a[:2] == ["kernels", "push"]:
    kf = opt("-p"); meta = json.load(open(os.path.join(kf, "kernel-metadata.json")))
    src = open(os.path.join(kf, meta["code_file"])).read().replace("/kaggle", os.path.join(root, "kaggle"))
    os.makedirs(state(meta["id"]), exist_ok=True); open(os.path.join(state(meta["id"]), "run.py"), "w").write(src)
    log = open(os.path.join(state(meta["id"]), "log"), "ab")
    p = subprocess.Popen([sys.executable, os.path.join(state(meta["id"]), "run.py")], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    open(os.path.join(state(meta["id"]), "pid"), "w").write(str(p.pid)); print("Kernel version 1 successfully pushed.")
elif a[:2] == ["kernels", "status"]:
    s = state(a[2]); pid = int(open(os.path.join(s, "pid")).read())
    try: os.kill(pid, 0); alive = not os.path.exists(os.path.join(root, "kaggle/working/sysone_job/outputs.tar.gz"))
    except OSError: alive = False
    st = "RUNNING" if alive else ("COMPLETE" if '"complete"' in open(os.path.join(root, "kaggle/working/sysone_job/outputs/status.json")).read() else "ERROR")
    print(f'{{a[2]}} has status "KernelWorkerStatus.{{st}}"')
elif a[:2] == ["kernels", "logs"]:
    s = state(a[-1]); pos = 0
    while True:
        data = open(os.path.join(s, "log"), "rb").read(); sys.stdout.write(data[pos:].decode("latin-1" if "-f" in a else "utf-8")); sys.stdout.flush(); pos = len(data)
        done = os.path.exists(os.path.join(root, "kaggle/working/sysone_job/outputs.tar.gz"))
        if "-f" not in a or done: sys.stdout.write(open(os.path.join(s, "log"), "rb").read()[pos:].decode("latin-1" if "-f" in a else "utf-8")); break
        time.sleep(0.5)
elif a[:2] == ["kernels", "output"]:
    dest = opt("-p"); os.makedirs(os.path.join(dest, "sysone_job"), exist_ok=True)
    shutil.copy(os.path.join(root, "kaggle/working/sysone_job/outputs.tar.gz"), os.path.join(dest, "sysone_job", "outputs.tar.gz"))
else: sys.exit(f"fake kaggle: {{a}} not implemented")
""")
(fake_bin / "kaggle").chmod(0o755)
os.environ |= {"PATH": f"{fake_bin}{os.pathsep}{os.environ['PATH']}", "FAKE_KAGGLE_ROOT": str(fake_kaggle), "KAGGLE_USERNAME": "me",
               "SYSONE_JOBS": str(Path(tempfile.mkdtemp()) / "jobs.json"), "SYSONE_JOBS_DIR": tempfile.mkdtemp()}

In [ ]:
script = Path(tempfile.mkdtemp()) / "train.py"
script.write_text("""import os
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, os.environ["ENCODER"], train="head", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3)
learn.export(job_output())
""")
kj = remote(script, on="kaggle", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": str(Path("fixtures/tiny-modernbert").resolve())},
            install=False)
test_eq(kj.ref, f"me/{kj.id}")
kj.wait(every=2, logs=False)
test_eq(kj.status, "complete")
assert [r for r in kj.progress() if r["event"] == "log"]          # followed live
assert "█" in kj.logs() and "â" not in kj.logs()                  # progress bars intact (see the finding below)
meta = json.loads((Path(kj.folder) / "kernel" / "kernel-metadata.json").read_text())
test_eq((meta["enable_gpu"], meta["enable_internet"], meta["is_private"], "machine_shape" in meta), (False, True, True, False))
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
test_eq(set(Decider.load(kj.fetch(), device="cpu").predict(rec["state"], rec["questions"])), set(rec["questions"]))
test_fail(lambda: kj.stop(), contains="kaggle.com/code/me/")

::: {.callout-note title="Finding: the Kaggle CLI garbles the text of a live log"}
`kaggle kernels logs -f` reads Kaggle's live log stream, which comes without a charset, so `requests` decodes its UTF-8 as Latin-1. On the first real kernel every `█` of a progress bar arrived as `â` and two invisible characters, and accented text would arrive the same way. Latin-1 maps each byte to one character, so writing that text back out as Latin-1 restores the original bytes: the follower runs the CLI with `PYTHONIOENCODING=latin-1`. The stand-in streams its log as the CLI does, and the test above finds its progress bars intact. Should the CLI decode UTF-8 one day, characters beyond Latin-1 would show as escapes, not garbled.
:::

In [ ]:
os.environ["PATH"] = os.environ["PATH"].split(os.pathsep, 1)[1]; del os.environ["FAKE_KAGGLE_ROOT"], os.environ["KAGGLE_USERNAME"]

The stand-in kernel ran with this machine's Python, which has sysone already (`install=False`). A real kernel gets the wheel of this checkout, packed into the runner with the rest of the job, and installs it first:

In [ ]:
wj = remote(script, on="kaggle", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"}, dry_run=True)
whl = json.loads((Path(wj.folder) / "job.json").read_text())["install"][0]
test_eq((whl.startswith("sysone-"), whl.endswith(".whl"), (Path(wj.folder) / whl).exists()), (True, True, True))

## Publishing to Kaggle Models

The Hub is where exports go by default; `publish_kaggle` also makes a Kaggle Model of an export, writing the two metadata files the CLI reads and uploading the folder as a variation (its subfolders zipped, which the CLI otherwise skips).

In [ ]:
#| export
def publish_kaggle(path, owner:str, model:str, variation:str="default", license:str="Apache 2.0", framework:str="pyTorch",
                   title:str|None=None, private:bool=True, dry_run:bool=False) -> dict:
    "Create a Kaggle Model and upload an export folder as one of its variations"
    path = Path(path)
    meta_dir = Path(tempfile.mkdtemp(prefix="sysone-kaggle-model-"))
    (meta_dir / "model-metadata.json").write_text(json.dumps({"ownerSlug": owner, "title": title or model, "slug": model,
        "subtitle": "", "isPrivate": private, "description": f"# {title or model}\n\nA typed-decision model exported by sysone.\n",
        "publishTime": "", "provenanceSources": ""}, indent=2))
    (path / "model-instance-metadata.json").write_text(json.dumps({"ownerSlug": owner, "modelSlug": model, "instanceSlug": variation,
        "framework": framework, "overview": "sysone export: sysone.json, head.safetensors, tokenizer/, encoder/",
        "usage": "from sysone.inference import Decider\nDecider.load(path).predict(state, questions)", "licenseName": license,
        "fineTunable": True, "trainingData": [], "modelInstanceType": "Unspecified", "baseModelInstance": "", "externalBaseModelUrl": ""}, indent=2))
    cmds = [["models", "create", "-p", str(meta_dir)], ["models", "variations", "create", "-p", str(path), "-r", "zip"]]
    if not dry_run:
        for c in cmds: run_kaggle(c, timeout=3600)
    return {"model": ["kaggle"] + cmds[0], "variation": ["kaggle"] + cmds[1]}

In [ ]:
exp = Path(tempfile.mkdtemp()); (exp / "sysone.json").write_text("{}")
cmds = publish_kaggle(exp, "me", "my-jev", dry_run=True)
test_eq(cmds["variation"][-2:], ["-r", "zip"])
test_eq(json.loads((exp / "model-instance-metadata.json").read_text())["instanceSlug"], "default")

## On real Kaggle

These cells need the Kaggle CLI and a login (`sysone.auth.login("kaggle")`); they are flagged `kaggle`, so the tests skip them unless run with `--flags kaggle`. The tiny training runs on a CPU kernel, then on two T4s.

In [ ]:
#| kaggle
real = remote(script, on="kaggle", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"})
real.wait(every=30)
test_eq(real.status, "complete")
Decider.load(real.fetch(), device="cpu").predict(rec["state"], rec["questions"])

In [ ]:
#| kaggle
t4 = remote(script, on="kaggle", hw="t4x2", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"})
t4.wait(every=30)
test_eq(t4.status, "complete")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()